[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/short_document_bias.ipynb)

# Does contacts-v1 generate "too-short" documents? — MarinFold [#142](https://github.com/Open-Athena/MarinFold/issues/142)

contacts-v1 predicts residue–residue contacts by **generating a document** of `<contact> <pI> <pJ>`
statements from the sequence alone. A natural worry: does the model **stop early** and emit **far
fewer contacts than the structure has**, capping recall no matter how good the inference wrapper is?

**Finding (spoiler).** Under-generation is real but *mild-to-moderate* — pooled `pred/gt ≈ 0.70`
(contacts emitted ÷ ground-truth count). Documents are **never truncated** (they emit `<end>` on
their own; a generous token budget never binds), and the shortfall **tracks difficulty**
(`corr(pred/gt, recall) = +0.84`): when the model knows the fold it emits near-complete documents
*with* good recall; when it's lost it emits a short document **and** the few contacts it does emit
are mostly wrong. So short documents are a *symptom* of uncertainty, not a decoding bug — strong
only on hard/long proteins. The dominant recall limiter is precision (wrong contacts), not brevity.

- **Part A** — no GPU (~30 s): reproduce the headline table + figure from the 2,400 published rollouts.
- **Part B** — GPU: generate your **own** rollouts from the model and watch the same pattern.

Model: the exp75 E8 1.5B (`prot-exp75-cv1-1_5b-e8-lr1e-3-wd0p2-v1-bc3084` step-35679, eval loss 2.7566),
the checkpoint the #142 analysis used. Part B also lets you swap in the newer default `contacts-v1-exp120-1.5B`.

## Part A — reproduce the published finding (no GPU)

Loads the exact run behind #142: **12 length-stratified eval proteins × 200 rollouts** (2,400 rows,
temperature 1.0 / top-p 0.95), scored against the canonical exp89 ground-truth contacts (degree>0,
sep≥6, resolved). Runs anywhere — Colab CPU is fine.

In [ ]:
import io, urllib.request
import numpy as np, pandas as pd, pyarrow.parquet as pq

RAW = "https://raw.githubusercontent.com/Open-Athena/MarinFold/06014e29c1f08cd7adea29f9d821ec0ac829c6c8/experiments/exp142_evals_short_document_bias"

def fetch_parquet(name):
    """Read a published parquet fixture anonymously (repo is public)."""
    with urllib.request.urlopen(f"{RAW}/{name}") as r:
        return pq.read_table(io.BytesIO(r.read())).to_pandas()

m = fetch_parquet("rollout_metrics_all.parquet")
m["pred_gt"] = m["n_pred"] / m["n_gt"]
p, r = m["all_prec"], m["all_rec"]
m["all_f1"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
print(f"{len(m):,} rollouts | {m.entry_id.nunique()} proteins | L {m.L.min()}-{m.L.max()}")

per = (m.groupby(["entry_id", "dataset", "L", "n_gt"])
         .agg(emitted=("n_pred", "mean"), pred_gt=("pred_gt", "mean"),
              frac_lt_half=("pred_gt", lambda s: (s < 0.5).mean()),
              tokens=("n_gen_tokens", "mean"), finished=("finished", "mean"),
              recall=("all_rec", "mean"), precision=("all_prec", "mean"),
              best_f1=("all_f1", "max"))
         .reset_index().sort_values("L"))
pd.set_option("display.width", 140)
print(per.round(3).to_string(index=False))

In [ ]:
# Pooled headline numbers
allr = m["pred_gt"].values
print(f"pooled pred/gt : mean {allr.mean():.2f}  median {np.median(allr):.2f} "
      f"(p10 {np.percentile(allr,10):.2f} - p90 {np.percentile(allr,90):.2f})")
print(f"rollouts emitting < half the GT contact count : {100*(allr<0.5).mean():.1f}%")
print(f"documents that finished (emitted <end>)        : {100*m['finished'].mean():.1f}%")
print(f"corr(pred/gt, L)      across proteins : {np.corrcoef(per.L, per.pred_gt)[0,1]:+.2f}")
print(f"corr(pred/gt, recall) across proteins : {np.corrcoef(per.pred_gt, per.recall)[0,1]:+.2f}"
      "   <- under-generation tracks accuracy => a symptom, not a decoding bug")

In [ ]:
import matplotlib.pyplot as plt

order = per.entry_id.tolist()
Ls = per.L.values
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# (1) pred/gt distribution per protein vs L
groups = [m.loc[m.entry_id == e, "pred_gt"].values for e in order]
bp = ax[0,0].boxplot(groups, positions=Ls, widths=8, showfliers=False,
                     patch_artist=True, manage_ticks=False)
for b in bp["boxes"]: b.set(facecolor="#4C72B0", alpha=0.5)
ax[0,0].axhline(1.0, color="crimson", ls="--", lw=1.5, label="parity (pred = GT count)")
ax[0,0].axhline(0.5, color="gray", ls=":", label="half of GT")
ax[0,0].set(xlabel="protein length L", ylabel="contacts emitted / n_gt", title="Contacts emitted vs ground truth")
ax[0,0].set_ylim(0, max(1.6, min(3.0, m.pred_gt.max()))); ax[0,0].legend(fontsize=8)

# (2) mean emitted vs n_gt (parity)
sc = ax[0,1].scatter(per.n_gt, per.emitted, c=Ls, cmap="viridis", s=70, zorder=3)
mx = max(per.n_gt.max(), per.emitted.max()) * 1.05
ax[0,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (parity)")
for _, row in per.iterrows():
    ax[0,1].annotate(str(int(row.L)), (row.n_gt, row.emitted), fontsize=7, xytext=(3,3), textcoords="offset points")
ax[0,1].set(xlabel="n_gt (ground-truth contacts)", ylabel="mean contacts emitted", title="Mean contacts emitted vs GT count")
ax[0,1].legend(fontsize=8); fig.colorbar(sc, ax=ax[0,1], label="L")

# (3) recall & precision vs L
ax[1,0].plot(Ls, per.recall, "o-", color="#4C72B0", label="recall (all, sep>=6)")
ax[1,0].plot(Ls, per.precision, "^--", color="#C44E52", label="precision (all)")
ax[1,0].set(xlabel="protein length L", ylabel="mean per-rollout metric", title="Rollout accuracy vs length", ylim=(0, None))
ax[1,0].legend(fontsize=8)

# (4) generated tokens vs complete-doc tokens
full = 3 * per.n_gt + 1
ax[1,1].scatter(full, per.tokens, c=Ls, cmap="viridis", s=70, zorder=3)
mx = max(full.max(), per.tokens.max()) * 1.05
ax[1,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (complete doc)")
ax[1,1].set(xlabel="tokens for a complete GT doc (3*n_gt+1)", ylabel="mean generated tokens", title="Document length: generated vs complete")
ax[1,1].legend(fontsize=8)

fig.suptitle("contacts-v1 1.5B (exp75 step-35679) - short-document-bias probe (12 eval proteins)", fontsize=12)
fig.tight_layout(); plt.show()

**Reading the figure.** Every box in panel 1 and every point in panels 2 & 4 sits **below parity** —
the model emits fewer contacts / shorter documents than the structure has. But the gap is modest for
most proteins and only blows up at the hard/long end (panel 2, the L=214 and L=426 points). Panel 3
shows accuracy collapsing on those same proteins: the short documents and the wrong contacts are the
*same* failure. The 100% finish rate (Part A printout) proves this is the model choosing `<end>`, not
a truncated generation.

## Part B — generate your own rollouts and verify (GPU)

Downloads the model from the public bucket, builds fresh contacts-v1 document realizations for a few
proteins, samples rollouts on the GPU, and shows the same `pred/gt` / `finished` pattern on **your**
samples. A Colab **T4** is plenty. (On CPU it still runs, just slowly — drop `N_ROLLOUTS`.)

In [ ]:
# @title Part B configuration
MODEL_NICKNAME = "contacts-v1-exp75-1.5B"  # @param ["contacts-v1-exp75-1.5B", "contacts-v1-exp120-1.5B"]
# short / confident-long / lost-long -> spans the range and shows the symptom contrast:
PROTEINS = "denovo_pdb__1uw1_A, denovo_pdb__6reo_A, foldbench100__8bfy_A"  # @param {type:"string"}
N_ROLLOUTS = 60  # @param {type:"integer"}
print(dict(model=MODEL_NICKNAME, proteins=[x.strip() for x in PROTEINS.split(",")], n_rollouts=N_ROLLOUTS))

In [ ]:
# Clone MarinFold + install marinfold[transformers] (build_document + model registry).
# marinfold pins transformers<5 so the Levanter rope config loads correctly.
import importlib, importlib.util, subprocess, sys
from pathlib import Path

REPO = Path("/content/MarinFold") if Path("/content").exists() else Path.cwd() / "MarinFold"
if not (REPO / "marinfold" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Open-Athena/MarinFold.git", str(REPO)], check=True)
if importlib.util.find_spec("marinfold") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-e", f"{REPO / 'marinfold'}[transformers]"], check=True)
    if str(REPO / "marinfold") not in sys.path:
        sys.path.insert(0, str(REPO / "marinfold"))
    importlib.invalidate_caches()
import marinfold
print("marinfold ready:", marinfold.__file__)

In [ ]:
import re, time
import numpy as np, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from marinfold.registry import resolve_model
from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, build_document, residues_from_sequence,
)

BEGIN, NUM_POS, MIN_SEP = "<begin_statements>", 2000, 6
CONTACT_RE = re.compile(r"<contact>\s+<p(\d+)>\s+<p(\d+)>")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
         else torch.float16 if device == "cuda" else torch.float32)

model_dir = resolve_model(MODEL_NICKNAME)          # downloads from the public bucket (anonymous)
tok = AutoTokenizer.from_pretrained(str(model_dir))
if tok.pad_token_id is None:
    tok.pad_token = "<end>"
model = AutoModelForCausalLM.from_pretrained(str(model_dir), dtype=dtype).to(device).eval()
END = tok.convert_tokens_to_ids("<end>")
print(f"loaded {MODEL_NICKNAME} on {device} ({dtype}); vocab={model.config.vocab_size}")

targets = fetch_parquet("targets.parquet").set_index("entry_id")   # sequences + resolved GT contacts


def prefix_and_map(entry, seq, r):
    """One fresh contacts-v1 realization (resampled N-term + statement order)."""
    doc = build_document(f"{entry}:r{r}", residues_from_sequence(seq), [], config=GenerationConfig())
    prefix = doc.document[: doc.document.index(BEGIN) + len(BEGIN)]
    pos2seq = {(doc.n_term_index + i) % NUM_POS: i for i in range(doc.seq_len)}
    return prefix, pos2seq


@torch.no_grad()
def rollouts_for(entry, n, batch=16 if torch.cuda.is_available() else 4):
    row = targets.loc[entry]
    seq, L, n_gt = row["sequence"], int(row["L"]), int(row["n_gt"])
    gt = {(int(i), int(j)) for i, j in row["gt_contacts"]}
    built = [prefix_and_map(entry, seq, r) for r in range(n)]
    plen = len(tok(built[0][0], add_special_tokens=False).input_ids)
    max_new = min(8192 - plen, 6 * L + 128)        # generous: cap never binds -> short == model's choice
    recs = []
    for s in range(0, n, batch):
        chunk = built[s:s + batch]
        enc = tok([c[0] for c in chunk], add_special_tokens=False,
                  return_tensors="pt", padding=True).to(device)
        out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                             do_sample=True, temperature=1.0, top_p=0.95, top_k=50,
                             max_new_tokens=max_new, eos_token_id=END, pad_token_id=END)
        gen = out[:, enc["input_ids"].shape[1]:]
        for (prefix, pos2seq), g in zip(chunk, gen.tolist()):
            finished = END in g
            g = g[: g.index(END) + 1] if finished else g
            pred = set()
            for a, b in CONTACT_RE.findall(tok.decode(g, skip_special_tokens=False)):
                ia, ib = pos2seq.get(int(a)), pos2seq.get(int(b))
                if ia is None or ib is None or abs(ia - ib) < MIN_SEP:
                    continue
                pred.add((min(ia, ib), max(ia, ib)))
            tp = len(pred & gt)
            recs.append(dict(entry=entry, L=L, n_gt=n_gt, n_pred=len(pred), finished=finished,
                             n_gen_tokens=len(g), recall=tp / len(gt) if gt else np.nan,
                             precision=tp / len(pred) if pred else 0.0))
    return recs


chosen = [p.strip() for p in PROTEINS.split(",") if p.strip()]
live = []
for e in chosen:
    t0 = time.time()
    rs = rollouts_for(e, N_ROLLOUTS)
    live += rs
    pg = np.mean([x["n_pred"] for x in rs]) / rs[0]["n_gt"]
    print(f"  {e:<24} L={rs[0]['L']:>3} n_gt={rs[0]['n_gt']:>3}  "
          f"pred/gt={pg:.2f}  finished={np.mean([x['finished'] for x in rs]):.2f}  "
          f"recall={np.nanmean([x['recall'] for x in rs]):.2f}  ({time.time()-t0:.0f}s)")
live = pd.DataFrame(live)

In [ ]:
# Your fresh rollouts vs the published 200-rollout run — same pattern?
liveagg = (live.assign(pred_gt=live.n_pred / live.n_gt)
               .groupby(["entry", "L", "n_gt"])
               .agg(pred_gt=("pred_gt", "mean"), finished=("finished", "mean"),
                    recall=("recall", "mean"), tokens=("n_gen_tokens", "mean"))
               .reset_index().sort_values("L"))
cmp = liveagg.merge(per[["entry_id", "pred_gt", "finished", "recall"]]
                    .rename(columns={"entry_id": "entry", "pred_gt": "pred_gt_pub",
                                     "finished": "finished_pub", "recall": "recall_pub"}),
                    on="entry", how="left")
print("your run (live) vs published (pub):")
print(cmp.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
groups = [live.loc[live.entry == e, "n_pred"].values / live.loc[live.entry == e, "n_gt"].values
          for e in liveagg.entry]
bp = ax.boxplot(groups, labels=[f"{e.split('__')[-1]}\nL={int(l)}" for e, l in zip(liveagg.entry, liveagg.L)],
                showfliers=False, patch_artist=True)
for b in bp["boxes"]: b.set(facecolor="#55A868", alpha=0.5)
ax.axhline(1.0, color="crimson", ls="--", label="parity"); ax.axhline(0.5, color="gray", ls=":")
ax.set(ylabel="contacts emitted / n_gt (your rollouts)", title=f"Your {N_ROLLOUTS} rollouts/protein — {MODEL_NICKNAME}")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Takeaway

You should see your live `pred/gt` and 100% finish rate line up with the published run: the model emits
~0.6–1.0× the GT contact count depending on the protein, always finishing on its own, with the shortfall
biggest exactly where recall is worst. Forcing longer documents (length penalty, banning early `<end>`,
lower temperature) would mostly add **wrong** contacts on the proteins that under-generate — so it isn't
the lever; the fix is a model that knows the fold.

- Issue & full writeup: [MarinFold #142](https://github.com/Open-Athena/MarinFold/issues/142)
- Code, data, and the offline analysis: [`experiments/exp142_evals_short_document_bias/`](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp142_evals_short_document_bias)